# NILM Model Training on Google Colab

Train a Non-Intrusive Load Monitoring (NILM) disaggregation model using the
`underscore-edge-model-pipeline` package. This notebook covers the full workflow: training,
evaluation, ONNX export, and downloading artefacts.

**Runtime:** Select *Runtime > Change runtime type > T4 GPU* before running.

## 1. Install

The repo is private, so clone it with a [personal access token](https://github.com/settings/tokens) stored in Colab Secrets as `GITHUB_TOKEN`.

In [1]:
import os
IS_HOSTED_COLAB = "COLAB_RELEASE_TAG" in os.environ
print("Hosted Colab" if IS_HOSTED_COLAB else "Local Colab runtime")

Hosted Colab


In [2]:
if IS_HOSTED_COLAB:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
else:
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "Set GITHUB_TOKEN via: docker run -e GITHUB_TOKEN=ghp_... ..."
        )

!git pull origin main #https://{token}@github.com/underscoreHQ/underscore-edge-model-pipeline.git
%pip install underscore-edge-model-pipeline/

fatal: not a git repository (or any of the parent directories): .git
Processing ./underscore-edge-model-pipeline
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for underscore-edge-model-pipeline: filename=underscore_edge_model_pipeline-0.1.0-py3-none-any.whl size=31270 sha256=bf46b8b2b2c92d12b2c8075f52ec1cfed5f0d316ade76e4e2b668d06218ac171
  Stored in directory: /root/.cache/pip/wheels/bc/6d/9d/80e4841712af2861f4d02d8858518ab273d06381e6123ae75c
Successfully built underscore-edge-model-pipeline
  Attempting uninstall: underscore-edge-model-pipeline
    Found existing installation: underscore-edge-model-pipeline 0.1.0
    Uninstalling underscore-edge-model-pipeline-0.1.0:
      Successfully uninstalled underscore-edge-model-pipeline-0.1.0


## 2. GPU check

In [3]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected — training will be slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

GPU: Tesla T4
Memory: 15.6 GB


## 3. Train

In [ ]:
import sys
sys.path.append('/content/underscore-edge-model-pipeline/src')

from edge_pipeline.training.trainer import train_model

model, trainer = train_model(
    model_type="tcn",
    window_size=60,
    batch_size=256,
    max_epochs=10,
    learning_rate=1e-3,
    use_amda=True,
    amda_scale=2.5,
    num_workers=2,
    output_dir="outputs",
    use_wandb=False,
    check_val_every_n_epoch=2,
)

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/outputs/checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss_fn    │ MSELoss    │      0 │ train │     0 │
│ 1 │ input_proj │ Conv1d     │    256 │ train │     0 │
│ 2 │ tcn        │ Sequential │  790 K │ train │     0 │
│ 3 │ fc         │ Sequential │ 17.2 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 807 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 807 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 98                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

## 4. Evaluate

In [ ]:
from edge_pipeline.data.datamodule import NilmDataModule
from edge_pipeline.evaluation.evaluate import APPLIANCE_NAMES, evaluate_model

# Set up a datamodule for evaluation (no augmentation)
eval_dm = NilmDataModule(
    window_size=60,
    batch_size=256,
    num_workers=2,
    use_amda=False,
)
eval_dm.setup()

device = "cuda" if torch.cuda.is_available() else "cpu"
results = evaluate_model(model, eval_dm, device=device)

print(f"{'Metric':<12} {'Value':>10}")
print("-" * 24)
for key in ["mae", "mse", "rmse", "r2", "nde"]:
    print(f"{key:<12} {results[key]:>10.4f}")

print(f"\n{'Appliance':<14} {'MAE':>10} {'R2':>10}")
print("-" * 36)
for name in APPLIANCE_NAMES:
    print(f"{name:<14} {results[f'{name}_mae']:>10.4f} {results[f'{name}_r2']:>10.4f}")

## 5. Export to ONNX

In [ ]:
from edge_pipeline.export.onnx_exporter import OnnxExporter

exporter = OnnxExporter(model, output_dir="exports", model_name="nilm_colab")
normalisation_metadata = eval_dm.get_normalisation_metadata()
onnx_path = exporter.export(normalisation_metadata=normalisation_metadata)
print(f"Exported ONNX model to {onnx_path}")

## 6. Download artefacts

In [ ]:
if IS_HOSTED_COLAB:
    from google.colab import files
    files.download(str(onnx_path))
    files.download("exports/nilm_colab_metadata.json")
else:
    print(f"Artefacts saved to:\n  {onnx_path}\n  exports/nilm_colab_metadata.json")
    print("Copy from the container with: docker cp <container>:/content/exports/ ./exports/")